# AgroUTMACH PIX
## NB01 — Preparación y auditoría del dataset PIO

**Objetivo:** descomprimir el material principal de PIO, verificar su estructura YOLO, comprobar la correspondencia imagen-etiqueta y visualizar ejemplos con sus *bounding boxes*.

**Entorno recomendado:** **CPU**.
Todavía no entrenaremos YOLO; la GPU se reservará para el notebook de entrenamiento.

> Regla del proyecto: los archivos originales de Zenodo permanecen intactos en `00_ZENODO_ORIGINAL`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, json, random, shutil, subprocess, sys

ROOT = Path('/content/drive/MyDrive/AgroUTMACH_PIX')
ZENODO = ROOT / '00_ZENODO_ORIGINAL'
DATASET = ROOT / '01_DATASET_PIO'
RESULTS = ROOT / '05_RESULTADOS'

PIO_OUT = DATASET / 'PIO_TRAIN_VAL'
TEST_OUT = DATASET / 'PIO_TEST_2025'

for p in [DATASET, RESULTS, PIO_OUT, TEST_OUT]:
    p.mkdir(parents=True, exist_ok=True)

required = {
    'data.rar': ZENODO / 'data.rar',
    'Test 2025.rar': ZENODO / 'Test 2025.rar',
    '0002.mp4': ZENODO / '0002.mp4',
    'Videos 2023.rar': ZENODO / 'Videos 2023.rar',
    'FilePrefixCode.xlsx': ZENODO / 'FilePrefixCode.xlsx',
    'Readme.txt': ZENODO / 'Readme.txt',
}

print('VERIFICACIÓN DE ARCHIVOS ORIGINALES')
print('-' * 60)
for name, path in required.items():
    if path.exists():
        print(f'✓ {name:22s}  {path.stat().st_size/1024**2:,.1f} MB')
    else:
        print(f'✗ FALTA: {name}')

assert required['data.rar'].exists(), 'No se encontró data.rar'
assert required['Test 2025.rar'].exists(), 'No se encontró Test 2025.rar'
print('\n✓ Archivos necesarios para NB01 encontrados.')

### 1. Instalar herramienta para archivos RAR
Ejecute esta celda una sola vez por sesión de Colab.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y unrar
print('✓ unrar instalado.')

### 2. Descomprimir `data.rar` y `Test 2025.rar`

La extracción se realiza en carpetas separadas para evitar mezclar el conjunto de entrenamiento/validación con el material de prueba de 2025.

Si las carpetas ya contienen archivos, el notebook no vuelve a extraerlas automáticamente.

In [ ]:
def folder_has_files(folder: Path):
    return folder.exists() and any(p.is_file() for p in folder.rglob('*'))

def extract_rar(rar_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    if folder_has_files(out_dir):
        print(f'↪ Ya hay archivos en {out_dir}. Se omite extracción.')
        return
    print(f'Extrayendo: {rar_path.name}')
    cmd = ['unrar', 'x', '-y', str(rar_path), str(out_dir) + '/']
    subprocess.run(cmd, check=True)
    print(f'✓ Extraído en: {out_dir}')

extract_rar(required['data.rar'], PIO_OUT)
extract_rar(required['Test 2025.rar'], TEST_OUT)

### 3. Mostrar la estructura encontrada

In [ ]:
def print_tree(root: Path, max_depth=4, max_items=120):
    root = root.resolve()
    print(f'\n{root.name}/')
    shown = 0
    for p in sorted(root.rglob('*')):
        try:
            rel = p.relative_to(root)
        except ValueError:
            continue
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = '    ' * (depth - 1)
        suffix = '/' if p.is_dir() else ''
        print(f'{indent}├── {p.name}{suffix}')
        shown += 1
        if shown >= max_items:
            print('    ... (salida truncada)')
            break

print_tree(PIO_OUT)
print_tree(TEST_OUT, max_depth=3, max_items=80)

### 4. Localizar automáticamente la raíz YOLO

Buscamos una carpeta que contenga simultáneamente `images/` y `labels/`.

In [ ]:
def find_yolo_roots(base: Path):
    candidates = []
    for p in [base] + [x for x in base.rglob('*') if x.is_dir()]:
        if (p / 'images').is_dir() and (p / 'labels').is_dir():
            candidates.append(p)
    return candidates

roots = find_yolo_roots(PIO_OUT)
print('Raíces YOLO candidatas:')
for i, r in enumerate(roots):
    print(f'[{i}] {r}')

assert roots, 'No se encontró una estructura con images/ y labels/.'
YOLO_ROOT = roots[0]
print(f'\n✓ Raíz YOLO seleccionada: {YOLO_ROOT}')

### 5. Auditoría de imágenes y etiquetas

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def image_files(folder: Path):
    return sorted([p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS])

def label_files(folder: Path):
    return sorted(folder.rglob('*.txt'))

audit = {}

for split in ['train', 'val']:
    img_dir = YOLO_ROOT / 'images' / split
    lbl_dir = YOLO_ROOT / 'labels' / split

    imgs = image_files(img_dir) if img_dir.exists() else []
    lbls = label_files(lbl_dir) if lbl_dir.exists() else []

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    missing_labels = sorted(img_stems - lbl_stems)
    orphan_labels = sorted(lbl_stems - img_stems)

    audit[split] = {
        'images': len(imgs),
        'labels': len(lbls),
        'images_without_label': len(missing_labels),
        'labels_without_image': len(orphan_labels),
    }

    print(f'\n{split.upper()}')
    print(f'  Imágenes:                 {len(imgs):,}')
    print(f'  Archivos de etiquetas:    {len(lbls):,}')
    print(f'  Imágenes sin etiqueta:    {len(missing_labels):,}')
    print(f'  Etiquetas sin imagen:     {len(orphan_labels):,}')

print('\nRESUMEN')
print(json.dumps(audit, indent=2, ensure_ascii=False))

### 6. Validar formato YOLO y contar instancias

Cada línea debe tener:

`class_id  x_center  y_center  width  height`

Las cuatro coordenadas normalizadas deben estar entre 0 y 1.

In [ ]:
def validate_labels(label_paths):
    total_instances = 0
    malformed = []
    class_counts = {}

    for lp in label_paths:
        with open(lp, 'r', encoding='utf-8', errors='ignore') as f:
            for nline, line in enumerate(f, start=1):
                s = line.strip()
                if not s:
                    continue
                parts = s.split()
                if len(parts) != 5:
                    malformed.append((str(lp), nline, s, 'n_columns'))
                    continue
                try:
                    cls = int(float(parts[0]))
                    vals = list(map(float, parts[1:]))
                except Exception:
                    malformed.append((str(lp), nline, s, 'parse'))
                    continue

                if not all(0.0 <= v <= 1.0 for v in vals):
                    malformed.append((str(lp), nline, s, 'range'))

                class_counts[cls] = class_counts.get(cls, 0) + 1
                total_instances += 1

    return total_instances, class_counts, malformed

all_labels = []
for split in ['train', 'val']:
    d = YOLO_ROOT / 'labels' / split
    if d.exists():
        all_labels.extend(label_files(d))

total_instances, class_counts, malformed = validate_labels(all_labels)

print(f'Instancias totales encontradas: {total_instances:,}')
print(f'Conteo por clase: {class_counts}')
print(f'Líneas problemáticas: {len(malformed):,}')

if malformed[:10]:
    print('\nPrimeros problemas:')
    for x in malformed[:10]:
        print(x)
else:
    print('✓ No se detectaron errores básicos de formato YOLO.')

### 7. Visualizar imágenes con sus anotaciones originales

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def find_label_for_image(img_path: Path, split: str):
    return YOLO_ROOT / 'labels' / split / f'{img_path.stem}.txt'

def draw_yolo_boxes(img_path: Path, label_path: Path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f'No se pudo abrir {img_path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if label_path.exists():
        with open(label_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = map(float, parts)
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255,255,255), 2)

    return img

train_imgs = image_files(YOLO_ROOT / 'images' / 'train')
sample_n = min(6, len(train_imgs))
sample = random.sample(train_imgs, sample_n)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = np.array(axes).ravel()

for ax in axes:
    ax.axis('off')

for ax, img_path in zip(axes, sample):
    lbl = find_label_for_image(img_path, 'train')
    img = draw_yolo_boxes(img_path, lbl)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

### 8. Localizar `dataset.yaml` original

In [ ]:
yamls = list(PIO_OUT.rglob('*.yaml')) + list(PIO_OUT.rglob('*.yml'))
print('Archivos YAML encontrados:')
for y in yamls:
    print(' -', y)

if yamls:
    print('\nContenido del primer YAML:')
    print('-' * 60)
    print(yamls[0].read_text(encoding='utf-8', errors='ignore'))
else:
    print('⚠ No se encontró dataset.yaml. En el siguiente notebook crearemos uno.')

### 9. Guardar reporte de auditoría

In [ ]:
report = {
    'project': 'AgroUTMACH PIX',
    'dataset': 'PIO',
    'yolo_root': str(YOLO_ROOT),
    'splits': audit,
    'total_instances': total_instances,
    'class_counts': class_counts,
    'malformed_label_lines': len(malformed),
    'yaml_files': [str(x) for x in yamls],
}

report_path = RESULTS / 'NB01_Auditoria_PIO.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f'✓ Reporte guardado en:\n{report_path}')
print('\nNB01 FINALIZADO CORRECTAMENTE.')
print('Siguiente etapa: NB02 — entrenamiento YOLO11 con GPU.')